# Advanced RAG

In [1]:
from langchain_community.document_loaders import GitLoader

def file_filter(file_path: str)->bool:
    return file_path.endswith(".md")

loader = GitLoader(
    clone_url="https://github.com/langchain-ai/langchain",
    repo_path="./langchain",
    branch="master",
    file_filter=file_filter
)

documents = loader.load()
print(len(documents))

28


In [2]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
import os
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

load_dotenv()
OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")

LANGSMITH_TRACING=os.getenv("LANGSMITH_TRACING")
LANGSMITH_ENDPOINT=os.getenv("LANGSMITH_ENDPOINT")
LANGSMITH_API_KEY=os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT=os.getenv("LANGSMITH_PROJECT")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
db = Chroma.from_documents(documents, embeddings)

In [3]:


prompt = ChatPromptTemplate.from_template('''
다음 문맥만을 고려해 질문에 답하세요.
문맥: """
{context}
"""

질문: {question}
''')

model = ChatOpenAI(model="gpt-5.4-nano", temperature=0)
retriever = db.as_retriever()

chain = {
    "question" : RunnablePassthrough(),
    "context" : retriever
} | prompt | model | StrOutputParser()

chain.invoke("LangChain의 개요를 알려줘")

'LangChain은 **LLM(대규모 언어 모델) 기반 에이전트와 애플리케이션을 만들기 위한 프레임워크**입니다. 다양한 구성요소와 **서드파티 통합(Integrations)** 을 연결해 AI 앱 개발을 단순화하고, 기술이 바뀌어도(underlying technology evolves) 선택을 “미래 지향적으로” 유지하도록 돕는 것을 목표로 합니다.\n\n- **모듈식으로 컴포넌트를 연결**해 LLM 애플리케이션을 구성할 수 있습니다.\n- **사전 구성된 에이전트 아키텍처/모델 통합**을 제공하여 빠르게 시작할 수 있게 합니다.\n- 필요에 따라 **LangGraph**(더 고급의 에이전트 오케스트레이션/런타임)를 함께 활용할 수 있습니다. LangChain 에이전트는 LangGraph 위에 구축되어 있습니다.\n- 개발/디버깅/배포에는 **LangSmith**를 권장하며, LLM 앱의 모니터링과 평가(evals), 디버깅을 지원하는 통합 플랫폼입니다.\n\n또한 문서에 따르면 LangChain을 사용하면 **모델/임베딩/벡터 스토어 등 표준 인터페이스**를 통해 개발할 수 있고, 데이터 소스와의 연결(실시간 데이터 증강), 모델 교체, 빠른 프로토타이핑, 프로덕션 기능(모니터링/평가/디버깅) 등의 장점이 있습니다.'

## 검색 쿼리 기법
### HyDE(Hypothetical Document Embeddings)
- 단순한 RAG 구성에서는 사용자의 질문에 대해 임베딩 벡터의 유사도가 높은 문서를 검색한다.
- 하지만 실제로 검색하고자 하는 것은 질문과 유사한 문서가 아니라, 답변과 유사한 문서이다. 이를 위해 HyDE라는 기법이 있다.
- HyDE에서는 사용자의 질문에 대해 LLM에 가상의 답변을 추론하게 하고, 그 출력을 임베딩 벡터의 유사도 검색에 사용한다.

In [4]:
# 가상의 답변을 생성하는 chain
hypothetical_prompt = ChatPromptTemplate.from_template("""
다음 질문에 한 문장으로 답하세요.
질문: {question}
""")

hypothetical_chain = hypothetical_prompt | model | StrOutputParser()

# 가상의 답변을 생성하는 Chain을 사용한 RAG
hyde_rag_chain = {
    "question": RunnablePassthrough(),
    "context" : hypothetical_chain | retriever # 가상의 답변을 생성하는 Chain의 출력을 retriever에 전달
} | prompt | model | StrOutputParser()

hyde_rag_chain.invoke("LangChain의 개요를 알려줘")

'LangChain은 **LLM(예: OpenAI, Anthropic, Google 등) 기반 에이전트와 애플리케이션을 만들기 위한 프레임워크**입니다. 여러 구성요소와 외부/서드파티 통합을 **연결(체인)**해서 AI 앱 개발을 단순화하고, 기반 기술이 변해도 쉽게 대응할 수 있도록 의사결정을 “미래 대비”할 수 있게 돕는 것을 목표로 합니다.\n\n- **에이전트/자율 애플리케이션 개발에 적합**: 소스 코드를 아주 적게 두고도 모델 제공자와 연결해 시작할 수 있습니다.\n- **사전 구성된 에이전트 아키텍처와 모델 통합**을 제공하여 빠르게 구축할 수 있습니다.\n- 더 고급 수준의 **제어 가능한 에이전트 워크플로우**가 필요하면, LangChain 위의 저수준 오케스트레이션 프레임워크인 **LangGraph**를 권장합니다.\n- 에코시스템으로는 **LangSmith(테스트/모니터링/디버깅)**, 다양한 **통합(Integrations)** 등이 함께 제공됩니다.\n\n원하시면, “LangChain 기본 개념” 관점에서 주요 구성요소(모델, 도구, 체인/에이전트 등)도 더 정리해드릴게요.'

### HypotheticalDocumentEmbedder
- 임베딩 단계에서 동작 - 질문을 하기전에 LLM으로 가상문서를 만들고, 그 가상문서를 임베딩 벡터로 변환 
- LCEL 방식의 hypothetical_chain + retriever 패턴이 langchain v1에서 권장되는 스타일이기 때문에 레거시 유틸 취급하여 `langchain_classic`에 위치함

### 복수 검색 쿼리 생성

In [5]:
from pydantic import BaseModel, Field

class QueryGenerationOutput(BaseModel):
    queries: list[str] = Field(..., description="검색 쿼리 목록")

query_generation_prompt = ChatPromptTemplate.from_template("""
질문에 대해 벡터 데이터베이스에서 관련 문서를 검색하기 위한 
3개의 서로 다른 검색 쿼리를 생성하세요.
거리 기반 유사성 검색의 한계를 극복하기 위해
사용자의 질문에 대해 여러 관점을 제공하는 것이 목표입니다.
질문: {question}
""")

query_generation_chain = (
    query_generation_prompt
    | model.with_structured_output(QueryGenerationOutput)
    | (lambda x : x.queries)
)

multi_query_rag_chain = {
    "question" : RunnablePassthrough(),
    "context" : query_generation_chain | retriever.map(),
} | prompt | model | StrOutputParser()

multi_query_rag_chain.invoke("LangChain의 개요를 알려줘")

'LangChain은 **LLM(대규모 언어 모델) 기반 에이전트와 애플리케이션을 만들기 위한 프레임워크**입니다. 서로 **연동 가능한 구성 요소와 서드파티 통합 기능**을 “연결(chain)”해서 AI 앱 개발을 쉽게 해주며, 기술이 바뀌더라도 의사결정을 **미래지향적으로 유지(future-proof)** 하도록 돕습니다.\n\n- **핵심 특징**\n  - LLM 공급자(예: OpenAI, Anthropic, Google 등)에 빠르게 연결\n  - **프리빌트 에이전트 아키텍처**와 모델 통합을 제공\n  - (기본 사용은) LangGraph를 몰라도 LangChain 에이전트를 활용 가능  \n    - LangChain의 에이전트는 **LangGraph 위에 구축**되어, durable 실행, 스트리밍, human-in-the-loop, persistence 등을 제공\n\n- **사용을 권장하는 경우**\n  - 에이전트 및 자율형(autonomous) 애플리케이션을 **빠르게 만들고 싶을 때**\n  - 더 고급 제어가 필요하면 **LangGraph**(저수준 에이전트 오케스트레이션/런타임)를 사용하라고 안내합니다.\n\n- **설치(예시)**\n  - `pip install langchain`\n\n- **문서/추가 도구**\n  - API 레퍼런스와 개념 가이드는 LangChain Docs/Reference에서 확인할 수 있습니다.\n  - 개발/디버깅/배포에는 **LangSmith**를 참고하라고 되어 있습니다.'

- 위 코드의 `retriever.map()`에서는 `list[str]`을 받아 `list[list[Document]]`를 반환하도록 변환. 

## 검색 후 기법
### RAG - Fusion
- 각 쿼리의 검색 결과를 프롬프트에 넣을 때, 특정 순서로 정렬할 필요가 있다.
- LangChain 공식 cookbook에서는 RRF 알고리즘을 사용한 `RAG-Fusion`기법을 소개한다.

In [9]:
from langchain_core.documents import Document

def reciprocal_rank_fusion(
        retriever_outputs: list[list[Document]],
        k: int = 60
):
    # 각 문서의 콘텐츠(문자열)와 그 점수의 매핑을 저장하는 딕셔너리 준비
    content_score_mapping = {}

    # 검색 쿼리마다 반복
    for docs in retriever_outputs:
        # 검색 결과의 문서마다 반복
        for rank, doc in enumerate(docs):
            content = doc.page_content

            # 처음 등장한 콘텐츠인 경우 0으로 초기화

            if content not in content_score_mapping:
                content_score_mapping[content] = 0
            
            # (1 / (순위 + k)) 점수를 추가
            content_score_mapping[content] += 1 / (rank + k)

    # 점수가 큰 순서로 정렬
    ranked = sorted(content_score_mapping.items(), key=lambda x: x[1], reverse=True)
    return [content for content, _ in ranked]


rag_fusion_chain = {
    "question": RunnablePassthrough(),
    "context": query_generation_chain | retriever.map() | reciprocal_rank_fusion
} | prompt | model | StrOutputParser()

rag_fusion_chain.invoke("LangChain의 개요를 알려줘")

'LangChain은 **LLM(대규모 언어 모델) 기반 에이전트와 애플리케이션을 만들기 위한 프레임워크**입니다. 여러 구성 요소와 **외부/제3자 통합(Integrations)** 을 “연결(chaining)”해서 AI 앱 개발을 더 쉽게 해주며, 기술이 바뀌어도 선택을 미래에 대비할 수 있도록(미래 대응) 추상화를 제공합니다.\n\n- **핵심 목적**: 모델, 임베딩, 벡터 스토어 등 다양한 구성요소를 표준 인터페이스로 묶어 LLM 앱 개발을 빠르게 진행\n- **실시간 데이터 보강(RAG 등)**: LLM을 다양한 데이터 소스/시스템과 연결\n- **모델 상호운용성**: 팀에서 다양한 모델을 실험해도 비교적 쉽게 교체 가능\n- **빠른 프로토타이핑**: 모듈형 구조로 접근/워크플로를 빠르게 바꿔가며 테스트\n- **프로덕션 기능**: 모니터링/평가/디버깅 같은 운영에 필요한 기능을 LangSmith 등과 함께 제공\n- **커뮤니티/생태계**: 다양한 통합, 템플릿, 커뮤니티 기여 컴포넌트 활용\n\n또한,\n- **LangGraph**: 더 고급 수준에서 “제어 가능한 에이전트 워크플로( deterministic + agentic 혼합, 지연시간 제어 등 )”를 만들 때 권장\n- **LangSmith**: AI 에이전트/LLM 앱 개발·디버깅·배포에 도움(관측/평가 포함)\n\n원하시면 “LangChain에서 에이전트/모델을 어떻게 연결하는지” 또는 “LangGraph와 차이”를 예시로 더 설명해드릴게요.'